# 🔀 Syndicate 3.0: Phase 2 Playroom (Concurrency & Streams)

Real-world agentic workflows often require making dozens of API calls to LLM endpoints, search engines, and databases. If you execute these calls sequentially (synchronously), your CPU sits idle waiting for network packets to return, slowing your system down to a crawl.

In this playroom, you will master **Concurrency** and **Real-Time Streaming** using Python's `asyncio` and `httpx` primitives.

---

## ⏳ 2.1: The Synchronous Bottleneck

#### 🛠 Built-in Module: System Latency Tracking (`import time`)
- The built-in `time` module maps Python variables directly to system clock ticks.
- We use `time.time()` to fetch epoch timestamps (seconds elapsed since January 1, 1970) to benchmark performance, and `time.sleep(seconds)` to suspend execution threads to simulate network latency.

### 🧪 Discovery Lab: Measuring Latency
Synchronous execution blocks the thread. While Python is waiting for a request to respond, no other line of code can execute. If one call takes 1 second, five calls will take 5 seconds.

In [ ]:
import time

def mock_sync_request(call_id: int) -> float:
    print(f"[SYNC] Call {call_id} starting...")
    time.sleep(1.0) # Simulating network latency
    print(f"[SYNC] Call {call_id} complete.")
    return 1.0

start_time = time.time()
# Run five mock requests sequentially
for i in range(5):
    mock_sync_request(i)
end_time = time.time()

print(f"\n[SYNC] Total time elapsed: {end_time - start_time:.4f} seconds")

## ⚡ 2.2: Asynchronous Concurrency (`asyncio` & `httpx`)
### Conceptual Lecture: Event Loop Coordination
Asynchronous code yields execution back to the **event loop** while waiting for I/O operations to complete. This allows the system to fire all requests in parallel, dropping the total execution time from $N 	imes T$ to roughly $\max(T)$.

#### 🛠 Built-in Module: The Concurrency Event Loop (`import asyncio`)
- The built-in `asyncio` engine operates an event loop that runs in a single system thread.
- When an asynchronous coroutine executes `await asyncio.sleep(1.0)`, it signals the loop that it is suspended waiting for external I/O. The loop immediately runs alternative queued tasks, maximizing throughput.

In [ ]:
import asyncio

async def mock_async_request(call_id: int) -> float:
    print(f"[ASYNC] Call {call_id} starting...")
    await asyncio.sleep(1.0) # Yielding flow back to the event loop
    print(f"[ASYNC] Call {call_id} complete.")
    return 1.0

async def run_concurrent():
    start_time = time.time()
    # TODO: Run 5 mock_async_requests concurrently using asyncio.gather
    # Replace the empty list below with the gathered tasks.
    results = []
    
    end_time = time.time()
    print(f"\n[ASYNC] Total time elapsed: {end_time - start_time:.4f} seconds")
    return results

# In a Jupyter notebook, the event loop is already running, so we await the coroutine directly:
results = await run_concurrent()
assert len(results) == 5, f"Expected 5 results, got {len(results)}"

## 🌊 2.3: Async Generators (`yield` & Streams)
### Conceptual Lecture: Slicing the Payload
Completions take time to generate. Waiting for the full completion to complete before showing output ruins the user experience. By using **async generators**, we can yield tokens the moment they are generated by the server, outputting text incrementally.

#### 🛠 Built-in Module & Typing: Custom Stream Writers (`import sys`, `AsyncGenerator`)
- In Python, the `print()` function automatically appends a newline character `\n` and buffers output in memory until a complete line is hit.
- To stream real-time tokens side-by-side, we bypass buffering using `sys.stdout.write(token)` (writes characters directly to the standard output buffer) and force an immediate screen update with `sys.stdout.flush()`.
- We type-hint the streaming pointer with `AsyncGenerator[YieldType, ReturnType]` from Python's standard `typing` library.

In [ ]:
from typing import AsyncGenerator
import sys

# A mocked token streamer
async def mock_token_stream(prompt: str) -> AsyncGenerator[str, None]:
    tokens = ["LLMs ", "are ", "remote ", "servers, ", "not ", "minds."]
    for token in tokens:
        await asyncio.sleep(0.3) # Simulating generation delay per token
        yield token

print("Starting stream: ")
# TODO: Consume the async generator mock_token_stream and print each token to the screen 
# as soon as it arrives without newline buffering (use sys.stdout.write and sys.stdout.flush).

# Your consumer loop here:

print("\nStream complete.")